# PlanetScope Image/tile Mosaic
- Yuwen Pang
- 8/7/2026
- PlanetScope imagery: aquifer_planetscope_202505_psscene_analytic_8b_sr_udm2

In [ ]:
from pathlib import Path
import re
import pandas as pd
import geopandas as gpd
import rasterio

# ============================================================
# 1. Configuration
# ============================================================

ra_dir = Path(
    r"W:\GIS_Data\SatelliteImagery\HighResolution\Morocco"
    r"\aquifer_planetscope_202505_psscene_analytic_8b_sr_udm2"
    r"\PSScene"
)

output_dir = ra_dir.parent / "Mosaics"
output_dir.mkdir(parents=True, exist_ok=True)

print("Input directory:", ra_dir)
print("Output directory:", output_dir)
print("Input directory exists:", ra_dir.exists())

aoi_path = Path(
    r"C:\Users\PangY\OneDrive - Smithsonian Institution\Bustard"
    r"\01_Data\Reneco\aquifer_limit\aquifer.shp"
)

aoi = gpd.read_file(aoi_path)

print("Number of AOI features:", len(aoi))
print("Original AOI CRS:", aoi.crs)
print("PlanetScope footprint CRS:", footprints.crs)

# only if the shapefile contains several polygons, you can combine them into one aquifer boundary:
# aoi_dissolved = aoi.dissolve()

In [ ]:
# 2. Search recursively in case each PlanetScope scene has its own folder
all_tifs = sorted(ra_dir.rglob("*.tif"))

# PlanetScope filenames may end with:
#   _3B_AnalyticMS_SR_8b.tif
#   _3B_AnalyticMS_SR_8b_clip.tif
#
# This expression includes both forms.
sr_pattern = re.compile(
    r"AnalyticMS_SR_8b(?:_clip)?\.tif$",
    flags=re.IGNORECASE
)

sr_files = [
    path for path in all_tifs
    if sr_pattern.search(path.name)
]

udm2_files = [
    path for path in all_tifs
    if "udm2" in path.name.lower()
]

print(f"Total TIFF files: {len(all_tifs)}")
print(f"8-band SR images: {len(sr_files)}")
print(f"UDM2 masks: {len(udm2_files)}")

for path in sr_files[:10]:
    print(path.name)

In [ ]:
# 3.Check how many acquisition dates are present

date_summary = (
    image_info
    .groupby("date", dropna=False)
    .size()
    .reset_index(name="image_count")
    .sort_values("date")
)

date_summary

In [ ]:
# 4. Validate CRS and resolution before mosaicking
unique_crs = image_info["crs"].dropna().unique()
unique_resolution = image_info[
    ["x_resolution", "y_resolution"]
].drop_duplicates()

if len(unique_crs) != 1:
    raise ValueError(
        "The images do not all have the same CRS. "
        "Reproject them before mosaicking.\n"
        f"CRS values found: {unique_crs}"
    )

if len(unique_resolution) != 1:
    print(
        "Warning: multiple pixel resolutions were found. "
        "GDAL will resample them during mosaic creation."
    )
    display(unique_resolution)
else:
    print("All images have the same CRS and pixel resolution.")

In [ ]:
def extract_planetscope_metadata(path):
    """
    Extract acquisition date, time, and scene identifiers
    from a typical PlanetScope filename.
    """

    parts = path.stem.split("_")

    date = pd.to_datetime(
        parts[0],
        format="%Y%m%d",
        errors="coerce"
    )

    time_text = parts[1] if len(parts) > 1 else None
    satellite_id = parts[3] if len(parts) > 3 else None

    return {
        "date": date,
        "date_text": date.strftime("%Y-%m-%d") if not pd.isna(date) else "Unknown",
        "time": time_text,
        "satellite": satellite_id
    }


footprint_records = []

for tile_number, path in enumerate(sr_files, start=1):

    metadata = extract_planetscope_metadata(path)

    with rasterio.open(path) as src:

        footprint_records.append({
            "tile_number": tile_number,
            "filename": path.name,
            "path": str(path),
            "date": metadata["date"],
            "date_text": metadata["date_text"],
            "time": metadata["time"],
            "satellite": metadata["satellite"],
            "bands": src.count,
            "crs": str(src.crs),
            "resolution_x": abs(src.transform.a),
            "resolution_y": abs(src.transform.e),
            "geometry": box(
                src.bounds.left,
                src.bounds.bottom,
                src.bounds.right,
                src.bounds.top
            )
        })


footprints = gpd.GeoDataFrame(
    footprint_records,
    geometry="geometry",
    crs="EPSG:32630"
)

footprints[
    [
        "tile_number",
        "date_text",
        "time",
        "satellite",
        "filename"
    ]
]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


def plot_tile_footprints(
    footprints,
    aoi=None,
    label_tiles=True,
    add_basemap=False,
    zoom_to="all",
    figsize=(13, 11)
):
    """
    Plot PlanetScope tile footprints and an optional AOI polygon.

    Parameters
    ----------
    footprints : GeoDataFrame
        PlanetScope tile polygons.

    aoi : GeoDataFrame, optional
        Area-of-interest polygon. It will be reprojected to the
        footprint CRS when necessary.

    label_tiles : bool
        Whether to label each tile with its tile number.

    add_basemap : bool
        Whether to add an OpenStreetMap basemap.

    zoom_to : {"all", "aoi"}
        "all" displays the full PlanetScope tile coverage.
        "aoi" zooms to the aquifer boundary with some padding.

    figsize : tuple
        Figure dimensions.
    """

    if footprints.empty:
        raise ValueError("The footprint GeoDataFrame is empty.")

    # Match the PlanetScope tile CRS
    aoi = aoi.to_crs(footprints.crs)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    unique_dates = sorted(
        footprints["date_text"].dropna().unique()
    )

    available_colors = [
        "tab:blue",
        "tab:orange",
        "tab:green",
        "tab:red",
        "tab:purple"
    ]

    date_colors = {
        date_text: available_colors[i % len(available_colors)]
        for i, date_text in enumerate(unique_dates)
    }

    # --------------------------------------------------------
    # Plot PlanetScope footprints
    # --------------------------------------------------------
    for date_text in unique_dates:

        subset = footprints[
            footprints["date_text"] == date_text
        ]

        subset.plot(
            ax=ax,
            facecolor=date_colors[date_text],
            edgecolor=date_colors[date_text],
            alpha=0.25,
            linewidth=1.3,
            zorder=2
        )

    # --------------------------------------------------------
    # Plot AOI
    # --------------------------------------------------------
    aoi_plot = None

    if aoi is not None:

        if aoi.empty:
            raise ValueError("The AOI GeoDataFrame is empty.")

        aoi_plot = aoi.copy()

        if aoi_plot.crs != footprints.crs:
            aoi_plot = aoi_plot.to_crs(footprints.crs)

        # Draw a light transparent fill and a strong boundary
        aoi_plot.plot(
            ax=ax,
            facecolor="none",
            edgecolor="black",
            linewidth=3.0,
            linestyle="--",
            zorder=5
        )

    # --------------------------------------------------------
    # Label individual tiles
    # --------------------------------------------------------
    if label_tiles:

        for _, row in footprints.iterrows():

            representative_point = (
                row.geometry.representative_point()
            )

            ax.text(
                representative_point.x,
                representative_point.y,
                str(row["tile_number"]),
                fontsize=8,
                ha="center",
                va="center",
                zorder=6,
                bbox={
                    "facecolor": "white",
                    "edgecolor": "black",
                    "linewidth": 0.3,
                    "alpha": 0.8,
                    "pad": 1.5
                }
            )

    # --------------------------------------------------------
    # Optional basemap
    # --------------------------------------------------------
    if add_basemap:

        try:
            import contextily as ctx

            ctx.add_basemap(
                ax,
                crs=footprints.crs.to_string(),
                source=ctx.providers.OpenStreetMap.Mapnik,
                zorder=1
            )

        except ImportError:
            print(
                "Contextily is not installed. Install it with:\n"
                "conda install -c conda-forge contextily"
            )

    # --------------------------------------------------------
    # Zoom to AOI if requested
    # --------------------------------------------------------
    if zoom_to == "aoi":

        if aoi_plot is None:
            raise ValueError(
                "zoom_to='aoi' requires an AOI GeoDataFrame."
            )

        xmin, ymin, xmax, ymax = aoi_plot.total_bounds

        x_padding = max((xmax - xmin) * 0.08, 100)
        y_padding = max((ymax - ymin) * 0.08, 100)

        ax.set_xlim(
            xmin - x_padding,
            xmax + x_padding
        )

        ax.set_ylim(
            ymin - y_padding,
            ymax + y_padding
        )

    elif zoom_to != "all":
        raise ValueError(
            "zoom_to must be either 'all' or 'aoi'."
        )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    legend_handles = [
        Patch(
            facecolor=date_colors[date_text],
            edgecolor=date_colors[date_text],
            alpha=0.35,
            label=date_text
        )
        for date_text in unique_dates
    ]

    if aoi_plot is not None:
        legend_handles.append(
            Line2D(
                [0],
                [0],
                color="black",
                linewidth=3,
                linestyle="--",
                label="Aquifer boundary"
            )
        )

    ax.legend(
        handles=legend_handles,
        title="Layers",
        loc="best"
    )

    ax.set_title(
        "PlanetScope tile footprints and aquifer boundary"
    )
    ax.set_xlabel("Easting, UTM Zone 30N (m)")
    ax.set_ylabel("Northing, UTM Zone 30N (m)")
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()

In [ ]:
# Remove Tile 22 while keeping the original footprints unchanged
footprints_clean = footprints.loc[
    footprints["tile_number"] != 22
].copy()

footprints_clean = footprints_clean.reset_index(drop=True)

In [ ]:
plot_tile_footprints(
    footprints=footprints_clean,
    aoi=aoi,
    label_tiles=True,
    add_basemap=False,
    zoom_to="aoi",
    figsize=(14, 11)
)

In [ ]:
# remove Tile 22 from the raster-file list
sr_files_clean = [
    Path(path)
    for path in footprints_clean["path"]
]

# Keep May 30 images later so they have mosaic priority
sr_files_clean = sorted(
    sr_files_clean,
    key=lambda path: path.name
)

print(f"Images for mosaic: {len(sr_files_clean)}")

for path in sr_files_clean:
    print(path.name)

In [ ]:
# 5. Memory-safe mosaic using GDAL VRT

from osgeo import gdal

gdal.UseExceptions()

def create_planetscope_mosaic(
    input_files,
    output_tif,
    src_nodata=0,
    output_nodata=0,
    compression="DEFLATE",
    predictor=2,
    overwrite=True
):
    """
    Create a memory-efficient PlanetScope mosaic using GDAL BuildVRT
    followed by GDAL Translate.

    Parameters
    ----------
    input_files : list of Path or str
        PlanetScope raster files to mosaic.

    output_tif : Path or str
        Output mosaic GeoTIFF.

    src_nodata : int or float, optional
        Nodata value in the source images. PlanetScope imagery often
        uses zero outside valid image footprints.

    output_nodata : int or float, optional
        Nodata value assigned to the output mosaic.

    compression : str
        GeoTIFF compression type, for example DEFLATE or LZW.

    predictor : int
        Predictor=2 is appropriate for integer reflectance values.

    overwrite : bool
        Whether to replace existing output files.
    """

    input_files = [str(Path(path)) for path in input_files]
    output_tif = Path(output_tif)
    output_tif.parent.mkdir(parents=True, exist_ok=True)

    if len(input_files) == 0:
        raise ValueError("No input files were provided.")

    vrt_path = output_tif.with_suffix(".vrt")

    if overwrite:
        for path in [output_tif, vrt_path]:
            if path.exists():
                path.unlink()

    print(f"Creating mosaic from {len(input_files)} images...")
    print("VRT:", vrt_path)
    print("Output:", output_tif)

    # Build a virtual raster.
    #
    # GDAL gives files later in the input list higher priority
    # where images overlap.
    vrt_options = gdal.BuildVRTOptions(
        resolution="highest",
        srcNodata=src_nodata,
        VRTNodata=output_nodata,
        resampleAlg="nearest"
    )

    vrt_dataset = gdal.BuildVRT(
        str(vrt_path),
        input_files,
        options=vrt_options
    )

    if vrt_dataset is None:
        raise RuntimeError("GDAL failed to create the VRT.")

    vrt_dataset.FlushCache()
    vrt_dataset = None

    # Convert VRT to a compressed tiled GeoTIFF
    translate_options = gdal.TranslateOptions(
        format="GTiff",
        noData=output_nodata,
        creationOptions=[
            "TILED=YES",
            f"COMPRESS={compression}",
            f"PREDICTOR={predictor}",
            "BIGTIFF=YES",
            "BLOCKXSIZE=512",
            "BLOCKYSIZE=512",
            "NUM_THREADS=ALL_CPUS"
        ]
    )

    output_dataset = gdal.Translate(
        str(output_tif),
        str(vrt_path),
        options=translate_options
    )

    if output_dataset is None:
        raise RuntimeError("GDAL failed to write the output mosaic.")

    output_dataset.FlushCache()
    output_dataset = None

    print("Mosaic completed successfully.")

    return output_tif

In [ ]:
mosaic_path = output_dir / "aquifer_planetscope_202505_mosaic.tif"

mosaic_path = create_planetscope_mosaic(
    # input_files=sr_files,
    input_files = sr_files_clean, #remove tile 22
    output_tif=mosaic_path,
    src_nodata=0,
    output_nodata=0
)

In [ ]:
with rasterio.open(mosaic_path) as src:
    print("Mosaic:", mosaic_path.name)
    print("Bands:", src.count)
    print("Dimensions:", src.width, "x", src.height)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Data types:", src.dtypes)
    print("Nodata:", src.nodata)
    print("Size on disk:", round(mosaic_path.stat().st_size / 1e9, 2), "GB")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pathlib import Path
from rasterio.enums import Resampling


def plot_planetscope_rgb(
    tif_path,
    rgb_bands=(6, 4, 2),
    max_display_dimension=2500,
    lower_percentile=2,
    upper_percentile=98,
    figsize=(13, 10),
    title=None
):
    """
    Plot a memory-efficient RGB preview of a PlanetScope 8-band raster.

    Parameters
    ----------
    tif_path : str or Path
        Input PlanetScope GeoTIFF.

    rgb_bands : tuple, default=(6, 4, 2)
        PlanetScope 8-band natural-color combination:
        Red=6, Green=4, Blue=2.

    max_display_dimension : int
        Maximum preview width or height in pixels.

    lower_percentile, upper_percentile : float
        Percentile stretch limits.

    figsize : tuple
        Matplotlib figure size.

    title : str, optional
        Custom plot title.
    """

    tif_path = Path(tif_path)

    if not tif_path.exists():
        raise FileNotFoundError(
            f"Raster not found:\n{tif_path}"
        )

    with rasterio.open(tif_path) as src:

        if src.count < max(rgb_bands):
            raise ValueError(
                f"Raster has {src.count} bands, but bands "
                f"{rgb_bands} were requested."
            )

        # Calculate downsampling scale
        scale = min(
            1.0,
            max_display_dimension / max(src.width, src.height)
        )

        display_width = max(
            1,
            int(src.width * scale)
        )

        display_height = max(
            1,
            int(src.height * scale)
        )

        # Read RGB bands at reduced resolution
        rgb = src.read(
            list(rgb_bands),
            out_shape=(
                3,
                display_height,
                display_width
            ),
            resampling=Resampling.bilinear
        ).astype("float32")

        # Read valid-data mask
        valid_mask = src.read_masks(
            rgb_bands[0],
            out_shape=(
                display_height,
                display_width
            ),
            resampling=Resampling.nearest
        ) > 0

        bounds = src.bounds
        crs = src.crs

    # Also exclude zero/nodata pixels
    valid_mask &= np.all(
        np.isfinite(rgb) & (rgb > 0),
        axis=0
    )

    rgb_stretched = np.zeros_like(
        rgb,
        dtype="float32"
    )

    # Stretch each RGB band independently
    for band_index in range(3):

        band = rgb[band_index]
        valid_values = band[valid_mask]

        if valid_values.size == 0:
            continue

        low, high = np.percentile(
            valid_values,
            [lower_percentile, upper_percentile]
        )

        if high <= low:
            continue

        rgb_stretched[band_index] = np.clip(
            (band - low) / (high - low),
            0,
            1
        )

    # Convert from bands × rows × columns to rows × columns × bands
    rgb_display = np.moveaxis(
        rgb_stretched,
        0,
        -1
    )

    # Use transparency for nodata
    alpha = np.where(
        valid_mask,
        1.0,
        0.0
    )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    ax.imshow(
        rgb_display,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ],
        origin="upper",
        alpha=alpha
    )

    if title is None:
        title = f"PlanetScope RGB mosaic: {tif_path.name}"

    ax.set_title(title)
    ax.set_xlabel("Easting")
    ax.set_ylabel("Northing")
    ax.set_aspect("equal")

    ax.text(
        0.01,
        0.01,
        f"CRS: {crs}",
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="bottom",
        bbox={
            "facecolor": "white",
            "alpha": 0.7,
            "edgecolor": "none"
        }
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_planetscope_rgb(
    mosaic_path,
    rgb_bands=(6, 4, 2),
    max_display_dimension=3000,
    lower_percentile=2,
    upper_percentile=98
)